# Data Download

This notebook downloads the requisite data needed to draft fantasy baseball players. It creates a data directory and stores the data that is easily and quickly digestable into PANDAS DataFrames

## Import libraries and parameters/options

In [1]:
from __future__ import annotations

from datetime import date
from pathlib import Path
from typing import Dict, List

import pandas as pd
from pybaseball import batting_stats, cache, pitching_stats

# Save pybaseball's internal cache to a local folder in this repo.
cache.enable()
cache.cache_directory(str(Path('data') / 'pybaseball_cache'))

BASE_DATA_DIR = Path('data')
RAW_DATA_DIR = BASE_DATA_DIR / 'raw'
PROCESSED_DATA_DIR = BASE_DATA_DIR / 'processed'

CURRENT_YEAR = date.today().year
YEARS_TO_PULL = [CURRENT_YEAR - 3, CURRENT_YEAR - 2, CURRENT_YEAR - 1]

# Toggle this to False if you only want CSV output.
WRITE_PARQUET = True


## Helper Functions

In [2]:
def ensure_directories() -> None:
    """Create local folders used by this notebook if they do not already exist."""
    for folder in [BASE_DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, BASE_DATA_DIR / 'pybaseball_cache']:
        folder.mkdir(parents=True, exist_ok=True)


def normalize_years(years: List[int]) -> List[int]:
    """Keep unique years in ascending order and exclude obviously invalid entries."""
    valid_years = sorted({year for year in years if 1871 <= year <= CURRENT_YEAR})
    if not valid_years:
        raise ValueError('No valid years provided. Update YEARS_TO_PULL with MLB seasons.')
    return valid_years


def download_season_data(year: int) -> Dict[str, pd.DataFrame]:
    """Download hitter and pitcher leaderboards for a single season."""
    print(f'Downloading data for {year}...')
    hitters_df = batting_stats(year, qual=0)
    pitchers_df = pitching_stats(year, qual=0)

    hitters_df['Season'] = year
    pitchers_df['Season'] = year

    return {
        'batting': hitters_df,
        'pitching': pitchers_df,
    }


def persist_dataframe(df: pd.DataFrame, dataset_name: str, year: int) -> Dict[str, Path]:
    """Write a dataframe to CSV (and optional Parquet) in ./data/raw."""
    csv_path = RAW_DATA_DIR / f'{dataset_name}_{year}.csv'
    df.to_csv(csv_path, index=False)

    paths = {'csv': csv_path}
    if WRITE_PARQUET:
        parquet_path = RAW_DATA_DIR / f'{dataset_name}_{year}.parquet'
        df.to_parquet(parquet_path, index=False)
        paths['parquet'] = parquet_path

    return paths


## Execute data download

In [3]:
ensure_directories()
years = normalize_years(YEARS_TO_PULL)

saved_files = []
all_batting: List[pd.DataFrame] = []
all_pitching: List[pd.DataFrame] = []

for year in years:
    season_data = download_season_data(year)

    batting_paths = persist_dataframe(season_data['batting'], 'batting_stats', year)
    pitching_paths = persist_dataframe(season_data['pitching'], 'pitching_stats', year)

    saved_files.append({'year': year, 'dataset': 'batting_stats', **{k: str(v) for k, v in batting_paths.items()}})
    saved_files.append({'year': year, 'dataset': 'pitching_stats', **{k: str(v) for k, v in pitching_paths.items()}})

    all_batting.append(season_data['batting'])
    all_pitching.append(season_data['pitching'])

batting_all_years = pd.concat(all_batting, ignore_index=True)
pitching_all_years = pd.concat(all_pitching, ignore_index=True)

batting_all_years.to_csv(PROCESSED_DATA_DIR / 'batting_stats_all_years.csv', index=False)
pitching_all_years.to_csv(PROCESSED_DATA_DIR / 'pitching_stats_all_years.csv', index=False)

if WRITE_PARQUET:
    batting_all_years.to_parquet(PROCESSED_DATA_DIR / 'batting_stats_all_years.parquet', index=False)
    pitching_all_years.to_parquet(PROCESSED_DATA_DIR / 'pitching_stats_all_years.parquet', index=False)

pd.DataFrame(saved_files)


## Test load for integrity

In [4]:
# Confirm we can load the local cache back into DataFrames for downstream analysis.
sample_batting_path = RAW_DATA_DIR / f'batting_stats_{years[-1]}.csv'
sample_pitching_path = RAW_DATA_DIR / f'pitching_stats_{years[-1]}.csv'

batting_check = pd.read_csv(sample_batting_path)
pitching_check = pd.read_csv(sample_pitching_path)

print(f'Loaded batting rows: {len(batting_check):,} from {sample_batting_path}')
print(f'Loaded pitching rows: {len(pitching_check):,} from {sample_pitching_path}')

batting_check.head(5)


## Output statistics and dataframe info

In [5]:
summary = pd.DataFrame(
    [
        {
            'dataset': 'batting_all_years',
            'rows': batting_all_years.shape[0],
            'columns': batting_all_years.shape[1],
            'years': f"{batting_all_years['Season'].min()}-{batting_all_years['Season'].max()}",
        },
        {
            'dataset': 'pitching_all_years',
            'rows': pitching_all_years.shape[0],
            'columns': pitching_all_years.shape[1],
            'years': f"{pitching_all_years['Season'].min()}-{pitching_all_years['Season'].max()}",
        },
    ]
)

summary


In [ ]:
# Optional: add more download sources (ADP/projections/injury feeds) in future iterations.
